# Regio-AI: Stage 1 — Satellite Data Exploration & Strandskydd Visualisation

**Project:** Fine-tuning IBM/NASA Prithvi for Strandskydd Violation Detection  
**Platform:** OpenShift AI — PyTorch notebook (CPU)  
**Data source:** Microsoft Planetary Computer — Sentinel-2 L2A (no credentials required)  
**Stage:** 1 of N — Real Satellite Data & Strandskydd Zone Derivation  

---

## Background

Sweden's *strandskydd* (shoreline protection law) prohibits construction within **100 metres of any water body** — lakes, rivers, and the sea. Enforcing this across Sweden's 500,000+ km of coastline and inland waterways is a major challenge for municipalities and county boards (*länsstyrelserna*).

This project explores using satellite imagery and the **IBM/NASA Prithvi geospatial foundation model** to automatically detect buildings that may violate strandskydd, flagging potential cases for human review.

## Stage 1 Goals

1. Set up the geospatial Python environment
2. Access real Sentinel-2 L2A imagery via Microsoft Planetary Computer (no credentials needed)
3. Define an Area of Interest (AOI) on the island of Orust, West Coast of Sweden
4. Visualise true-colour RGB and compute NDWI water detection from real data
5. Derive a strandskydd 100m protection zone from the satellite-derived water mask
6. Build an interactive map combining all layers

---

## 1. Environment Setup

Install required packages on top of the OpenShift AI PyTorch base image. Run once per notebook server session.

In [ ]:
%pip install --quiet \
    planetary-computer \
    pystac-client \
    stackstac \
    rasterio \
    geopandas \
    shapely \
    folium \
    pyproj \
    numpy \
    matplotlib \
    pillow
print("Installation complete.")

## 2. Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import folium
from folium import plugins
from shapely.geometry import shape
import rasterio
from rasterio.features import shapes as rasterio_shapes
from affine import Affine
import pystac_client
import planetary_computer
import stackstac

print(f"geopandas   : {gpd.__version__}")
print(f"folium      : {folium.__version__}")
print(f"stackstac   : {stackstac.__version__}")
print("All imports successful.")

## 3. Configuration — Area of Interest & Parameters

We focus on **Orust** — Sweden's third largest island on the west coast (*Bohuslän*). The area has extensive coastline, numerous inlets and skerries, and a dense mix of year-round and summer residences, making it a highly relevant area for strandskydd analysis.

All coordinates are **WGS84 (EPSG:4326)**.

In [ ]:
# --- Area of Interest ---
AOI_NAME = "Orust, Bohuslän, West Coast of Sweden"

# Bounding box centred on the point of interest (~15 km × 19 km)
AOI_BBOX = {
    "west":  11.65,
    "south": 58.18,
    "east":  11.91,
    "north": 58.35
}

AOI_CENTER = [
    (AOI_BBOX["south"] + AOI_BBOX["north"]) / 2,
    (AOI_BBOX["west"]  + AOI_BBOX["east"])  / 2
]

# [west, south, east, north] list — used by STAC and stackstac
BBOX_LIST = [AOI_BBOX["west"], AOI_BBOX["south"], AOI_BBOX["east"], AOI_BBOX["north"]]

# --- Point of Interest (specific location to investigate) ---
POI_LAT  = 58.26599
POI_LON  = 11.77902
POI_LABEL = "Summer house location — suspected new structure post-2019"

# --- Strandskydd ---
STRANDSKYDD_DISTANCE_M = 100  # metres from any water body (standard zone)

# --- Sentinel-2 temporal extents ---
# Prithvi expects 6 HLS-compatible bands; we use the closest Sentinel-2 equivalents
PRITHVI_BANDS       = ["B02", "B03", "B04", "B8A", "B11", "B12"]  # Blue, Green, Red, Narrow NIR, SWIR1, SWIR2
MAX_CLOUD_COVER     = 20   # %
TEMPORAL_AFTER      = ["2022-06-01", "2023-09-30"]  # After new structure built
TEMPORAL_BEFORE     = ["2017-01-01", "2018-12-31"]  # Before new structure

print(f"AOI          : {AOI_NAME}")
print(f"Bounding box : {AOI_BBOX}")
print(f"POI          : {POI_LAT}°N, {POI_LON}°E")
print(f"Strandskydd  : {STRANDSKYDD_DISTANCE_M} m from water")
print(f"Bands        : {PRITHVI_BANDS}")
print(f"After period : {TEMPORAL_AFTER}")
print(f"Before period: {TEMPORAL_BEFORE}")


## 4. Sentinel-2 Data Access via Microsoft Planetary Computer

[Microsoft Planetary Computer](https://planetarycomputer.microsoft.com) hosts Sentinel-2 Level-2A imagery as Cloud-Optimised GeoTIFFs (COGs) on Azure, freely accessible without authentication. The `planetary-computer` Python package handles URL signing automatically.

We use the STAC (SpatioTemporal Asset Catalog) standard to search and retrieve scenes.

In [ ]:
PLANETARY_COMPUTER_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"

catalog = pystac_client.Client.open(
    PLANETARY_COMPUTER_URL,
    modifier=planetary_computer.sign_inplace,
)

print(f"Connected to : {PLANETARY_COMPUTER_URL}")
print(f"Catalog      : {catalog.title}\n")

# Search for 'after' scenes (post new structure)
search_after = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=BBOX_LIST,
    datetime=f"{TEMPORAL_AFTER[0]}/{TEMPORAL_AFTER[1]}",
    query={"eo:cloud_cover": {"lt": MAX_CLOUD_COVER}},
)
items_after = search_after.item_collection()
print(f"'After' scenes  : {len(items_after)}  (cloud < {MAX_CLOUD_COVER}%)")

# Search for 'before' scenes (pre-2019)
search_before = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=BBOX_LIST,
    datetime=f"{TEMPORAL_BEFORE[0]}/{TEMPORAL_BEFORE[1]}",
    query={"eo:cloud_cover": {"lt": MAX_CLOUD_COVER}},
)
items_before = search_before.item_collection()
print(f"'Before' scenes : {len(items_before)}  (cloud < {MAX_CLOUD_COVER}%)\n")

for label, items in [("AFTER", items_after), ("BEFORE", items_before)]:
    print(f"--- {label} ---")
    for item in sorted(items, key=lambda x: x.properties["eo:cloud_cover"])[:5]:
        date  = item.datetime.strftime("%Y-%m-%d")
        cloud = item.properties["eo:cloud_cover"]
        tile  = item.properties.get("s2:mgrs_tile", "—")
        print(f"  {date}  tile: {tile}  cloud: {cloud:.1f}%")
    print()


## 5. Load Best Scene

We pick the scene with the lowest cloud cover. `stackstac` creates a **lazy Dask array** — the data stays on Planetary Computer's Azure storage until we call `.compute()`, so this cell is fast.

In [ ]:
if not items_after:
    raise RuntimeError("No 'after' scenes found. Adjust TEMPORAL_AFTER or MAX_CLOUD_COVER.")
if not items_before:
    raise RuntimeError("No 'before' scenes found. Adjust TEMPORAL_BEFORE or MAX_CLOUD_COVER.")

# Pick the least cloudy scene from each period
best_after  = min(items_after,  key=lambda x: x.properties["eo:cloud_cover"])
best_before = min(items_before, key=lambda x: x.properties["eo:cloud_cover"])

date_after  = best_after.datetime.strftime("%Y-%m-%d")
date_before = best_before.datetime.strftime("%Y-%m-%d")

print(f"'After'  scene : {date_after}   cloud: {best_after.properties['eo:cloud_cover']:.1f}%")
print(f"'Before' scene : {date_before}  cloud: {best_before.properties['eo:cloud_cover']:.1f}%\n")

# Build lazy DataArrays — nothing downloaded yet
def make_stack(item):
    return stackstac.stack(
        [item],
        assets=PRITHVI_BANDS,
        bounds_latlon=BBOX_LIST,
        resolution=10,
        dtype="float64",
        fill_value=np.nan,
        rescale=False,
        epsg=32633,  # UTM Zone 33N — confirmed from tile T33VUE
    )

da_after  = make_stack(best_after)
da_before = make_stack(best_before)

print(f"Dimensions : {dict(zip(da_after.dims, da_after.shape))}")
print(f"CRS        : {da_after.attrs.get('crs', 'unknown')}")
print(f"Resolution : 10 m/pixel")


## 6. Download & True-Colour RGB

`.compute()` triggers the actual download from Planetary Computer. The data is clipped to our AOI so only the pixels we need are fetched.

In [ ]:
print("Downloading 'after' scene...")
data_after = da_after.squeeze("time").compute()
print(f"  {date_after} — {data_after.nbytes / 1e6:.1f} MB")

print("Downloading 'before' scene...")
data_before = da_before.squeeze("time").compute()
print(f"  {date_before} — {data_before.nbytes / 1e6:.1f} MB\n")

# Convenience alias — 'after' is the primary scene for NDWI / strandskydd
data     = data_after
date_str = date_after
cloud_pct = best_after.properties["eo:cloud_cover"]


def extract_rgb(d):
    """Return a display-ready RGB composite from a DataArray."""
    def stretch(arr, p_low=2, p_high=98):
        valid = arr[np.isfinite(arr) & (arr > 0)]
        if len(valid) == 0:
            return np.zeros_like(arr, dtype=float)
        lo, hi = np.percentile(valid, [p_low, p_high])
        return np.clip((arr - lo) / (hi - lo + 1e-10), 0, 1)
    return np.dstack([
        stretch(d.sel(band="B04").values),
        stretch(d.sel(band="B03").values),
        stretch(d.sel(band="B02").values),
    ])


rgb_after  = extract_rgb(data_after)
rgb_before = extract_rgb(data_before)
rgb        = rgb_after  # alias used in later cells

# Extract 'after' bands (used for NDWI and strandskydd)
B02 = data_after.sel(band="B02").values
B03 = data_after.sel(band="B03").values
B04 = data_after.sel(band="B04").values
B8A = data_after.sel(band="B8A").values
B11 = data_after.sel(band="B11").values
B12 = data_after.sel(band="B12").values

# Side-by-side RGB preview
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
axes[0].imshow(rgb_before)
axes[0].set_title(f"BEFORE — {date_before}", fontsize=13, color="steelblue")
axes[0].axis("off")
axes[1].imshow(rgb_after)
axes[1].set_title(f"AFTER  — {date_after}", fontsize=13, color="firebrick")
axes[1].axis("off")
plt.suptitle(f"Sentinel-2 True Colour RGB — {AOI_NAME}", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()
print(f"Image size: {rgb_after.shape[1]} × {rgb_after.shape[0]} px  |  10 m/pixel")


## 7. NDWI Water Detection

The **Normalised Difference Water Index (NDWI)** uses the Green and Near-Infrared bands to isolate open water:

$$\text{NDWI} = \frac{\text{Green} - \text{NIR}}{\text{Green} + \text{NIR}}$$

Values **> 0.1** are classified as water (the slightly elevated threshold reduces noise from cloud shadows and wet soil).

In [ ]:
def compute_ndwi(green: np.ndarray, nir: np.ndarray) -> np.ndarray:
    """Normalised Difference Water Index. Values >0 indicate water."""
    g, n = green.astype(float), nir.astype(float)
    denom = g + n
    denom[denom == 0] = np.nan
    return (g - n) / denom


ndwi       = compute_ndwi(B03, B8A)
water_mask = ndwi > 0.1

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(rgb)
axes[0].set_title("True Colour RGB", fontsize=12)
axes[0].axis("off")

im1 = axes[1].imshow(ndwi, cmap="RdYlBu", vmin=-0.5, vmax=0.5)
axes[1].set_title("NDWI\n(blue = water, red = land / vegetation)", fontsize=12)
axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

water_pct = water_mask.mean() * 100
axes[2].imshow(water_mask, cmap="Blues")
axes[2].set_title(
    f"Water Mask (NDWI > 0.1)\n{water_mask.sum():,} px  |  {water_pct:.1f}% of scene",
    fontsize=12
)
axes[2].axis("off")

plt.suptitle(
    f"NDWI Water Detection — {AOI_NAME} — {date_str}",
    fontsize=13, y=1.01
)
plt.tight_layout()
plt.show()

## 8. Strandskydd 100 m Buffer

We vectorise the water mask raster into polygons, dissolve them, and apply a **100 m buffer** in the native UTM projection (metres). The result is reprojected to WGS84 for the interactive map.

This is the strandskydd protection zone derived entirely from satellite data — no pre-existing boundary data needed.

In [ ]:
# --- Resolve CRS from stackstac attrs ---
crs_attr = data.attrs.get("crs")
if crs_attr is None:
    utm_crs = "EPSG:32633"  # UTM Zone 33N — covers most of Sweden
elif hasattr(crs_attr, "to_epsg"):
    utm_crs = f"EPSG:{crs_attr.to_epsg()}"
else:
    utm_crs = str(crs_attr)

# --- Resolve affine transform ---
scene_transform = data.attrs.get("transform")
if scene_transform is None:
    # Reconstruct from spatial coordinates
    x_vals = data.x.values
    y_vals = data.y.values
    dx = float(x_vals[1] - x_vals[0])
    dy = float(y_vals[1] - y_vals[0])   # negative (top → bottom)
    scene_transform = Affine(
        dx,  0, float(x_vals[0]) - dx / 2,
         0, dy, float(y_vals[0]) - dy / 2
    )

print(f"CRS       : {utm_crs}")
print(f"Transform : {scene_transform}\n")

# --- Vectorise water pixels ---
water_uint8 = water_mask.astype(np.uint8)
water_geoms = [
    shape(geom)
    for geom, val in rasterio_shapes(water_uint8, transform=scene_transform)
    if val == 1
]
print(f"Raw water polygons : {len(water_geoms):,}")

strandskydd_gdf = None
water_wgs84     = None

if water_geoms:
    water_gdf = gpd.GeoDataFrame(geometry=water_geoms, crs=utm_crs)

    # Dissolve → simplify (20 m tolerance) → reproject for map
    water_dissolved = water_gdf.dissolve()
    water_wgs84 = (
        water_dissolved
        .simplify(20)
        .to_frame("geometry")
        .set_crs(utm_crs)
        .to_crs("EPSG:4326")
    )

    # Buffer → reproject
    strandskydd_gdf = (
        gpd.GeoDataFrame(
            geometry=water_dissolved.buffer(STRANDSKYDD_DISTANCE_M),
            crs=utm_crs
        )
        .to_crs("EPSG:4326")
    )

    area_km2 = strandskydd_gdf.to_crs(utm_crs).area.sum() / 1e6
    print(f"Strandskydd buffer : {STRANDSKYDD_DISTANCE_M} m — OK")
    print(f"Protected area     : {area_km2:.1f} km²")
else:
    print("No water polygons found. Try lowering the NDWI threshold in cell 7.")

## Change Detection — Before vs After

We compare the two Sentinel-2 scenes using the **Normalised Difference Built-up Index (NDBI)**:

$$\text{NDBI} = \frac{\text{SWIR}_1 - \text{NIR}}{\text{SWIR}_1 + \text{NIR}}$$

Higher NDBI values indicate built-up or bare surfaces. A **positive NDBI change** (after − before) highlights areas that became more built-up between the two dates — a strong signal of new construction.

The red marker on the map shows the specific location of interest: `58.26599°N, 11.77902°E`.

In [ ]:
def compute_ndbi(swir1: np.ndarray, nir: np.ndarray) -> np.ndarray:
    """Normalised Difference Built-up Index. Higher = more built-up / bare soil."""
    s, n = swir1.astype(float), nir.astype(float)
    denom = s + n
    denom[denom == 0] = np.nan
    return (s - n) / denom


B8A_before = data_before.sel(band="B8A").values
B11_before = data_before.sel(band="B11").values

ndbi_after  = compute_ndbi(B11,        B8A)
ndbi_before = compute_ndbi(B11_before, B8A_before)
ndbi_change = ndbi_after - ndbi_before   # positive = new built-up surface

# NDWI change (negative = water lost / land gained)
B03_before = data_before.sel(band="B03").values
ndwi_before = compute_ndwi(B03_before, B8A_before)
ndwi_change = compute_ndwi(B03, B8A) - ndwi_before

# ---- visualise ----
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

axes[0, 0].imshow(rgb_before)
axes[0, 0].set_title(f"RGB — BEFORE ({date_before})", fontsize=11, color="steelblue")
axes[0, 0].axis("off")

axes[0, 1].imshow(rgb_after)
axes[0, 1].set_title(f"RGB — AFTER ({date_after})", fontsize=11, color="firebrick")
axes[0, 1].axis("off")

diff_rgb = np.abs(rgb_after.astype(float) - rgb_before.astype(float)).mean(axis=2)
im_diff = axes[0, 2].imshow(diff_rgb, cmap="hot", vmin=0, vmax=0.3)
axes[0, 2].set_title("RGB Difference magnitude", fontsize=11)
axes[0, 2].axis("off")
plt.colorbar(im_diff, ax=axes[0, 2], fraction=0.046, pad=0.04)

im_nb = axes[1, 0].imshow(ndbi_before, cmap="RdYlGn_r", vmin=-0.4, vmax=0.4)
axes[1, 0].set_title(f"NDBI — BEFORE ({date_before})", fontsize=11)
axes[1, 0].axis("off")
plt.colorbar(im_nb, ax=axes[1, 0], fraction=0.046, pad=0.04)

im_na = axes[1, 1].imshow(ndbi_after, cmap="RdYlGn_r", vmin=-0.4, vmax=0.4)
axes[1, 1].set_title(f"NDBI — AFTER ({date_after})", fontsize=11)
axes[1, 1].axis("off")
plt.colorbar(im_na, ax=axes[1, 1], fraction=0.046, pad=0.04)

im_ch = axes[1, 2].imshow(ndbi_change, cmap="seismic", vmin=-0.3, vmax=0.3)
axes[1, 2].set_title(
    "NDBI Change (After − Before)\n"
    "red = new built-up  |  blue = new vegetation/water",
    fontsize=11
)
axes[1, 2].axis("off")
plt.colorbar(im_ch, ax=axes[1, 2], fraction=0.046, pad=0.04)

plt.suptitle(
    f"Change Detection — {AOI_NAME}\n"
    f"{date_before}  →  {date_after}",
    fontsize=13, y=1.01
)
plt.tight_layout()
plt.show()

# Summary stats
threshold = 0.05
new_built = (ndbi_change > threshold) & np.isfinite(ndbi_change)
print(f"Pixels with NDBI increase > {threshold} (new built-up): {new_built.sum():,}")
print(f"Approx. area of new built-up surfaces: {new_built.sum() * 100 / 1e4:.2f} ha")


## 9. Interactive Map

Combining all layers:

| Layer | Source |
|---|---|
| Satellite imagery | Esri World Imagery (public tiles) |
| Area of Interest | Configured bounding box |
| Water bodies | NDWI-derived from real Sentinel-2 data |
| Strandskydd zone | 100 m buffer computed from satellite water mask |
| Strandskydd (official) | Naturvårdsverket WMS (authoritative reference) |

In [ ]:
m = folium.Map(
    location=AOI_CENTER,
    zoom_start=12,
    tiles=None,
    control_scale=True
)

# --- Base layers ---
folium.TileLayer(
    tiles=(
        "https://server.arcgisonline.com/ArcGIS/rest/services/"
        "World_Imagery/MapServer/tile/{z}/{y}/{x}"
    ),
    attr="Esri World Imagery",
    name="Satellite (Esri)",
    overlay=False,
    control=True
).add_to(m)

folium.TileLayer(
    "OpenStreetMap",
    name="OpenStreetMap",
    overlay=False,
    control=True
).add_to(m)

# --- AOI bounding box ---
folium.Rectangle(
    bounds=[
        [AOI_BBOX["south"], AOI_BBOX["west"]],
        [AOI_BBOX["north"], AOI_BBOX["east"]]
    ],
    color="#FFD700", weight=2,
    fill=True, fill_color="#FFD700", fill_opacity=0.05,
    tooltip=f"Area of Interest: {AOI_NAME}",
    name="Area of Interest"
).add_to(m)

# --- NDWI-derived water bodies ---
if water_wgs84 is not None:
    folium.GeoJson(
        water_wgs84.__geo_interface__,
        name=f"Water bodies (Sentinel-2 NDWI, {date_str})",
        style_function=lambda x: {
            "fillColor": "#3399FF",
            "color":      "#0066CC",
            "weight":     1,
            "fillOpacity": 0.5,
        },
        tooltip=f"Water body — NDWI > 0.1 from {date_str}"
    ).add_to(m)

# --- Satellite-derived strandskydd buffer ---
if strandskydd_gdf is not None:
    folium.GeoJson(
        strandskydd_gdf.__geo_interface__,
        name=f"Strandskydd zone — {STRANDSKYDD_DISTANCE_M} m (satellite-derived)",
        style_function=lambda x: {
            "fillColor": "#FF4444",
            "color":      "#CC0000",
            "weight":     1,
            "fillOpacity": 0.25,
        },
        tooltip=f"Protected zone — {STRANDSKYDD_DISTANCE_M} m from water (strandskydd)"
    ).add_to(m)

# --- Official Naturvårdsverket WMS (authoritative reference) ---
folium.WmsTileLayer(
    url="https://nvpub.vic-metria.nu/arcgis/services/Strandskydd/MapServer/WmsServer",
    name="Strandskydd — Naturvårdsverket (official WMS)",
    layers="0",
    fmt="image/png",
    transparent=True,
    overlay=True,
    control=True,
    opacity=0.6,
    attr="Naturvårdsverket"
).add_to(m)


# --- Point of Interest marker ---
folium.Marker(
    location=[POI_LAT, POI_LON],
    popup=folium.Popup(POI_LABEL, max_width=300),
    tooltip="Point of Interest — click for details",
    icon=folium.Icon(color="red", icon="home", prefix="fa")
).add_to(m)

# 500m zoom circle around POI for context
folium.Circle(
    location=[POI_LAT, POI_LON],
    radius=500,
    color="red",
    weight=1.5,
    fill=False,
    tooltip="500 m radius around POI"
).add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
plugins.MiniMap(toggle_display=True).add_to(m)
plugins.Fullscreen().add_to(m)

print(f"Map ready — {AOI_NAME}")
print(f"Scene date  : {date_str}  (cloud cover: {cloud_pct:.1f}%)")
print("Layers      : AOI | Water bodies (NDWI) | Strandskydd buffer | NV WMS | Satellite")
m

## 10. Summary & Stage 2 Roadmap

### Stage 1 Complete

| Step | Status |
|---|---|
| Geospatial environment | Ready |
| Planetary Computer access | Working — no credentials needed |
| Sentinel-2 L2A download | Real data |
| True-colour RGB | Done |
| NDWI water detection | Done — real satellite data |
| Strandskydd 100 m buffer | Computed from satellite water mask |
| Interactive map | All layers combined |

---

### Stage 2 — Building Detection with IBM/NASA Prithvi

With the full data pipeline working, Stage 2 will add:

1. **Load Prithvi-100M** from HuggingFace (`ibm-nasa-geospatial/Prithvi-100M`)
2. **Preprocess** the 6-band Sentinel-2 tile into HLS-compatible normalised input tensors
3. **Attach a segmentation head** and run inference on CPU to detect building footprints
4. **Intersect** detected buildings with the NDWI-derived strandskydd buffer
5. **Flag and visualise** potential strandskydd violations on the interactive map

The 6 bands loaded in this notebook (`B02, B03, B04, B8A, B11, B12`) are already the correct Prithvi input bands — no changes to the data pipeline needed.

---

### Resources

- [Microsoft Planetary Computer](https://planetarycomputer.microsoft.com)
- [Prithvi-100M on HuggingFace](https://huggingface.co/ibm-nasa-geospatial/Prithvi-100M)
- [Naturvårdsverket geodata](https://www.naturvardsverket.se/verktyg-och-tjanster/kartor-och-geografisk-information/)
- [Lantmäteriet building data (TopoDB)](https://www.lantmateriet.se)
- [stackstac documentation](https://stackstac.readthedocs.io)